# チェン探索（Chien Search）によるエラー位置の特定

<a href="https://github.com/Tsukumo-999/qr-code-from-scratch/blob/master/Step3/3_chien_search.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

第10回記事で行った、BM法で得られたエラー位置多項式 $\sigma(x)$ にガロア体の要素を総当たりで代入し、エラーの場所を突き止めるチェン探索の実験ノートブックです。

### 1. ガロア体の準備とBM法の結果の引き継ぎ
前回計算した受信データと、BM法で得られたエラー位置多項式 `C = [1, 40, 29]` をセットします。

In [2]:
# ガロア体 GF(2^8) の準備
exp_table = [0] * 256
log_table = [0] * 256
v = 1
for i in range(255):
    exp_table[i] = v
    log_table[v] = i
    v <<= 1
    if v & 0x100: v ^= 0x11D
exp_table[255] = exp_table[0]

def gf_mul(x, y):
    if x == 0 or y == 0: return 0
    return exp_table[(log_table[x] + log_table[y]) % 255]

# 意図的に2箇所（インデックス1と3）を壊した受信メッセージ長 7バイト
received_message = [72, 99, 76, 200, 162, 112, 246]
msg_length = len(received_message)

# 前回のBM法で得られたエラー位置多項式 σ(x) = 1 + 40x + 29x^2
C = [1, 40, 29]

print(f"受信データ: {received_message}")
print(f"エラー位置多項式 C(x): {C}")

受信データ: [72, 99, 76, 200, 162, 112, 246]
エラー位置多項式 C(x): [1, 40, 29]


### 2. チェン探索（Chien Search）の実装
メッセージの長さ（7要素分）だけ、多項式 $C(x)$ に $\alpha^{-i}$ を順番に代入して計算結果が 0 になるかを調べます。
（※ $\alpha^{-i}$ はガロア体において $\alpha^{255-i}$ と同じ意味になります）

In [3]:
print("【チェン探索による総当たり開始】\n")

error_positions = [] # エラーが起きた位置（i）を保存するリスト
error_indexes = []   # 配列上での実際のインデックス

# メッセージの長さ分だけループ (i = 0 から 6)
for i in range(msg_length):
    # 代入する x = α^{-i} を計算
    # i=0 の時は x=1(α^0)、それ以外は 255-i 乗
    x_val = 1 if i == 0 else exp_table[255 - i]
    
    # 多項式 C(x) に x_val を代入して計算（ホーナー法などを用いない単純な代入計算）
    # C(x) = C[0] + C[1]*x + C[2]*x^2 + ...
    result = 0
    x_pow = 1 # x^0
    
    for coef in C:
        # result += coef * x^k (ガロア体上の加算はXOR)
        result ^= gf_mul(coef, x_pow)
        # 次の項のために x を掛けて x^(k+1) を作る
        x_pow = gf_mul(x_pow, x_val)
        
    print(f"  [検証 i={i}] x に α^{'-'+str(i) if i>0 else '0'} を代入 -> 計算結果: {result}")
    
    # 計算結果が 0 なら、そこがエラーの場所！
    if result == 0:
        print("    🎯 ゼロになりました！ここがエラー位置です！")
        error_positions.append(i)
        # 配列のインデックス（左から何番目か）に変換する
        idx = (msg_length - 1) - i
        error_indexes.append(idx)

print("\n====================================")
print(f"🔍 探索完了！ エラーが発見されたインデックス: {error_indexes}")
print("====================================")

【チェン探索による総当たり開始】

  [検証 i=0] x に α^0 を代入 -> 計算結果: 52
  [検証 i=1] x に α^-1 を代入 -> 計算結果: 85
  [検証 i=2] x に α^-2 を代入 -> 計算結果: 27
  [検証 i=3] x に α^-3 を代入 -> 計算結果: 0
    🎯 ゼロになりました！ここがエラー位置です！
  [検証 i=4] x に α^-4 を代入 -> 計算結果: 140
  [検証 i=5] x に α^-5 を代入 -> 計算結果: 0
    🎯 ゼロになりました！ここがエラー位置です！
  [検証 i=6] x に α^-6 を代入 -> 計算結果: 250

🔍 探索完了！ エラーが発見されたインデックス: [3, 1]


### 3. 次回予告（フォルニー法へ）
見事に、私たちが意図的に壊した「インデックス1」と「インデックス3」を計算の力だけで特定することができました。

あとは、特定された場所の数字が「どれだけ化けてしまったか（エラー値）」を算出し、反転（XOR）させて元通りに修復するだけです。
次回、復元編「フォルニー（Forney）法」に続きます！